In [ ]:
!pip install --upgrade pip setuptools wheel -q
!pip install --upgrade cmake -q
!pip install scs --prefer-binary -q
!pip install cvxpy --prefer-binary -q
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary
!pip install catboost


In [ ]:
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from optbinning import BinningProcess
import shutil
from warnings import simplefilter
simplefilter(action = "ignore") #, category = FutureWarning

pd.set_option('display.max_rows', 500)
from sklearn.preprocessing import LabelEncoder


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple
#import plotly.graph_objects as go
#from plotly.subplots import make_subplots
import os
#import plotly.express as px

pd.set_option('display.float_format', '{:.2f}'.format)


In [ ]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")


In [ ]:
%%time
query = """
WITH mst_plaft_trx_operaciones AS (

    SELECT
        cod_argcatalogo AS cod_trx,
        desc_argcatalogo AS des_trx,
        desc_valor02 AS tipo_transaccion,
        desc_valor03 AS actor

    FROM e_perm_aws.t_mst_parametros_plaft
    WHERE desc_catalogo = 'TIPO OPERACION'

),

base_operaciones AS (

    SELECT

        SUBSTR(o.fec_inicio_trx,1,4) ||
        SUBSTR(o.fec_inicio_trx,6,2) AS periodo,

        CASE
            WHEN p.actor = 'ORD'
                THEN o.codunico_ord
            ELSE o.codunico_ben
        END AS id_origen,

        CASE
            WHEN p.actor = 'ORD'
                THEN o.codunico_ben
            ELSE o.codunico_ord
        END AS id_destino,

        o.cod_trx_orig,
        p.des_trx,
        p.tipo_transaccion,

        o.mto_operacion,
        o.fec_inicio_trx

    FROM e_perm_aws.t_aml_operaciones o

    INNER JOIN mst_plaft_trx_operaciones p
        ON o.cod_trx_orig = p.cod_trx

    WHERE o.codunico_ord <> o.codunico_ben

)

SELECT

    periodo,

    id_origen,
    id_destino,

    cod_trx_orig,
    des_trx,
    tipo_transaccion,

    COUNT(*) AS cantidad_operaciones,

    SUM(mto_operacion) AS monto_total,

    AVG(mto_operacion) AS monto_promedio,

    MAX(mto_operacion) AS monto_maximo,

    MIN(fec_inicio_trx) AS fecha_primera_operacion,

    MAX(fec_inicio_trx) AS fecha_ultima_operacion

FROM base_operaciones

WHERE periodo =  '202604'

GROUP BY
    periodo,
    id_origen,
    id_destino,
    cod_trx_orig,
    des_trx,
    tipo_transaccion
;"""
df = athena_query(query, database='disc_comercial')
df.head()


In [ ]:

df.head()


In [ ]:
%%time
query = """
WITH base_cucibk AS (

    SELECT
        cod_cuc,
        key_value
    FROM (

        SELECT
            cod_cuc,
            key_value,
            ROW_NUMBER() OVER(
                PARTITION BY cod_cuc
                ORDER BY fch_priprd DESC
            ) rn

        FROM e_perm_aws.t_mst_ctaibk_rsk

    )
    WHERE rn = 1

),

cliente_pep AS (

    SELECT
        b.cod_cuc AS codunico,
        1 AS flag_pep,
        MAX(r3.cargo) AS cargo_pep,
        MAX(r3.sbs_fecha_inicio) AS fecha_inicio_pep,
        MAX(r3.sbs_fecha_cese) AS fecha_fin_pep

    FROM base_cucibk b

    INNER JOIN e_perm_aws.t_listas_nivel_de_riesgos_motivo_r3 r3
        ON b.key_value = r3.key_value

    GROUP BY 1

),

alertas AS (

    SELECT

        codunico,

        MAX(
            CASE
                WHEN fecha_inicio_ros IS NOT NULL
                THEN 1
                ELSE 0
            END
        ) AS flag_ros,

        MAX(
            CASE
                WHEN fecha_inicio_desvinculacion IS NOT NULL
                THEN 1
                ELSE 0
            END
        ) AS flag_desv,

        MAX(
            CASE
                WHEN periodo_alerta IS NOT NULL
                THEN 1
                ELSE 0
            END
        ) AS flag_alerta,

        COUNT(
            DISTINCT CASE
                WHEN fecha_inicio_ros IS NOT NULL
                THEN fecha_inicio_ros
            END
        ) AS cnt_ros,

        COUNT(
            DISTINCT periodo_alerta
        ) AS cnt_alertas

    FROM e_perm_aws.t_alertas_plaft

    GROUP BY 1

)

SELECT

    cp.codunico AS node_id,

    cp.tipo_cliente_sensible_vinc,

    cp.nivel_riesgo_pep,

    cp.nivel_riesgo_not_neg,

    cp.score_con_excepcion_a,

    cp.cantidad_noticias,

    cp.cantidad_lsb,

    cp.cantidad_oficios,

    COALESCE(pep.flag_pep,0) AS flag_pep,

    pep.cargo_pep,

    pep.fecha_inicio_pep,

    pep.fecha_fin_pep,

    COALESCE(a.flag_ros,0) AS flag_ros,

    COALESCE(a.flag_alerta,0) AS flag_alerta,

    COALESCE(a.flag_desv,0) AS flag_desv,

    COALESCE(a.cnt_ros,0) AS cnt_ros,

    COALESCE(a.cnt_alertas,0) AS cnt_alertas,

    CASE
        WHEN cp.cantidad_noticias > 0
          OR cp.cantidad_not_asociado_laft > 0
          OR cp.cantidad_not_1a_alto > 0
          OR cp.cantidad_not_5a_alto > 0
        THEN 1
        ELSE 0
    END AS flag_noticia

FROM e_perm_aws.t_cliente_plaft cp

LEFT JOIN cliente_pep pep
    ON cp.codunico = pep.codunico

LEFT JOIN alertas a
    ON cp.codunico = a.codunico;"""
df_1 = athena_query(query, database='disc_comercial')
df_1.head()


In [ ]:
%%time
query = """
WITH pd AS (
    SELECT DISTINCT
        -- Identificadores
        a.key_value,
        a.cod_cli,
        date_format(
            date_parse(CAST(a.cod_mes AS varchar), '%Y%m') - interval '1' month,
            '%Y%m'
        ) AS codmes_lag1,
        CAST(a.cod_mes AS INTEGER) AS cod_mes
    FROM d_perm_aws.t_agg_alertas_plaft a
        WHERE a.cod_mes = '202604'
            AND a.desc_subsegmento = 'BPE'
),

target AS (
    SELECT 
        codunico,
        periodo_alerta,
        tipo_alerta_n2 as tip_alerta,
        trx_riesgo_cliente,
        MAX(calificacion_monitoreo) AS flg_alerta
    FROM e_perm_aws.t_alertas_plaft
    GROUP BY codunico, periodo_alerta, tipo_alerta_n2,trx_riesgo_cliente
)

SELECT 
    a.key_value,
    a.cod_cli,
    a.codmes_lag1,
    a.cod_mes,
    b.tip_alerta,
    b.trx_riesgo_cliente, 
    CASE 
        WHEN b.flg_alerta = '1' THEN 1 
        ELSE 0 
    END AS target
FROM pd a
LEFT JOIN target b
    ON a.cod_cli = b.codunico
    AND cast(a.cod_mes as varchar) = b.periodo_alerta
--   AND codmes_lag1 = c.periodo_alerta;"""
alertas_p1 = athena_query(query, database='disc_comercial')
alertas_p1.head()


In [ ]:
alertas_p1.target.value_counts()


In [ ]:
import pandas as pd

alertas = pd.read_csv(
    r"C:\Users\b46637\OneDrive - Interbank\PLAFT\Interbank\PLAFT\Desarrollo\Minorista_regulado\Inferencia_manual\score_202607_minorista.csv"
)


In [ ]:
alertas.head()


In [ ]:
alertas_p1 = alertas[
    alertas["grupo_score"].eq("P5") &
    (pd.to_numeric(alertas["score"], errors="coerce") > 0.98)
].copy()

print("Clientes P5 con score > 0.98:", alertas_p1.shape)


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx


In [ ]:
# =========================
# 1. COPIAS DE TRABAJO
# =========================

trx = df.copy()
nodos = df_1.copy()
p1 = alertas_p1.copy()

# Normalizar nombres
trx.columns = trx.columns.str.strip().str.lower()
nodos.columns = nodos.columns.str.strip().str.lower()
p1.columns = p1.columns.str.strip().str.lower()

print("trx:", trx.shape)
print("nodos:", nodos.shape)
print("P5 score > 0.98:", p1.shape)


In [ ]:
# =========================
# 2. NORMALIZAR IDs
# =========================

def normalizar_id(serie):
    return (
        serie
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(10)
    )


trx["origen_id"] = normalizar_id(trx["id_origen"])
trx["destino_id"] = normalizar_id(trx["id_destino"])

nodos["node_id"] = normalizar_id(nodos["node_id"])

p1["cod_cli"] = normalizar_id(p1["cod_cli"])


In [ ]:
print(trx["origen_id"].head().tolist())
print(nodos["node_id"].head().tolist())
print(p1["cod_cli"].head().tolist())


In [ ]:
# =========================
# 3. FLAGS PLAFT
# =========================

flags = [
    "flag_pep",
    "flag_ros",
    "flag_alerta",
    "flag_noticia"
]

for col in flags:
    if col in nodos.columns:
        nodos[col] = (
            pd.to_numeric(
                nodos[col],
                errors="coerce"
            )
            .fillna(0)
            .astype("int8")
        )


In [ ]:
for col in flags:
    if col in nodos.columns:
        print("\n", col)
        print(nodos[col].value_counts(dropna=False))


In [ ]:
# =========================
# 4. LIMPIEZA TRANSACCIONES
# =========================

trx = trx[
    trx["origen_id"].notna() &
    trx["destino_id"].notna()
].copy()

# Excluir auto-relaciones
trx = trx[
    trx["origen_id"] != trx["destino_id"]
].copy()

# IDs técnicos / inválidos
ids_invalidos = {
    "0000000000"
}

trx = trx[
    ~trx["origen_id"].isin(ids_invalidos) &
    ~trx["destino_id"].isin(ids_invalidos)
].copy()


In [ ]:
columnas_numericas = [
    "cantidad_operaciones",
    "monto_total",
    "monto_promedio",
    "monto_maximo"
]

for col in columnas_numericas:
    if col in trx.columns:
        trx[col] = (
            pd.to_numeric(
                trx[col],
                errors="coerce"
            )
            .fillna(0)
        )


In [ ]:
for col in [
    "fecha_primera_operacion",
    "fecha_ultima_operacion"
]:
    if col in trx.columns:
        trx[col] = pd.to_datetime(
            trx[col],
            errors="coerce"
        )


In [ ]:
# =========================
# 5. AGREGAR ARISTAS
# =========================

edges = (
    trx
    .groupby(
        ["periodo", "origen_id", "destino_id"],
        as_index=False
    )
    .agg(
        cantidad_operaciones=(
            "cantidad_operaciones",
            "sum"
        ),

        monto_total=(
            "monto_total",
            "sum"
        ),

        monto_maximo=(
            "monto_maximo",
            "max"
        ),

        cantidad_tipos_trx=(
            "cod_trx_orig",
            "nunique"
        ),

        fecha_primera_operacion=(
            "fecha_primera_operacion",
            "min"
        ),

        fecha_ultima_operacion=(
            "fecha_ultima_operacion",
            "max"
        )
    )
)


In [ ]:
edges["monto_promedio"] = (
    edges["monto_total"] /
    edges["cantidad_operaciones"].replace(0, np.nan)
)


In [ ]:
print("Aristas:", len(edges))

edges.head()


In [ ]:
# =========================
# 6. GRAFO DIRIGIDO
# =========================

G = nx.DiGraph()

for row in edges.itertuples(index=False):

    G.add_edge(
        row.origen_id,
        row.destino_id,

        periodo=row.periodo,

        cantidad_operaciones=row.cantidad_operaciones,
        monto_total=row.monto_total,
        monto_promedio=row.monto_promedio,
        monto_maximo=row.monto_maximo,
        cantidad_tipos_trx=row.cantidad_tipos_trx,

        fecha_primera_operacion=row.fecha_primera_operacion,
        fecha_ultima_operacion=row.fecha_ultima_operacion
    )


In [ ]:
print(f"Nodos:   {G.number_of_nodes():,}")
print(f"Aristas: {G.number_of_edges():,}")


In [ ]:
# =========================
# 7. ATRIBUTOS PLAFT
# =========================

nodos_unique = (
    nodos
    .dropna(subset=["node_id"])
    .drop_duplicates(
        subset=["node_id"],
        keep="last"
    )
)


In [ ]:
atributos_nodos = (
    nodos_unique
    .set_index("node_id")
    .to_dict(orient="index")
)

nx.set_node_attributes(
    G,
    atributos_nodos
)


In [ ]:
sum(
    attrs.get("flag_ros", 0) == 1
    for _, attrs in G.nodes(data=True)
)


In [ ]:
# =========================
# 8. P5 SCORE > 0.98 EN EL GRAFO
# =========================

clientes_p1 = set(
    p1["cod_cli"]
    .dropna()
    .unique()
)

clientes_p1_grafo = [
    cli
    for cli in clientes_p1
    if cli in G
]

print(f"P5 score > 0.98 totales:            {len(clientes_p1):,}")
print(f"P5 score > 0.98 presentes en grafo: {len(clientes_p1_grafo):,}")

print(
    "Cobertura:",
    round(
        100 * len(clientes_p1_grafo) /
        len(clientes_p1),
        2
    ),
    "%"
)


In [ ]:
# =========================
# 9. MÉTRICAS CLIENTE
# =========================

def metricas_cliente(G, cliente):

    if cliente not in G:
        return None

    sucesores = list(G.successors(cliente))
    predecesores = list(G.predecessors(cliente))

    vecinos = set(sucesores) | set(predecesores)

    monto_enviado = sum(
        G[cliente][v].get("monto_total", 0)
        for v in sucesores
    )

    monto_recibido = sum(
        G[v][cliente].get("monto_total", 0)
        for v in predecesores
    )

    operaciones_enviadas = sum(
        G[cliente][v].get("cantidad_operaciones", 0)
        for v in sucesores
    )

    operaciones_recibidas = sum(
        G[v][cliente].get("cantidad_operaciones", 0)
        for v in predecesores
    )

    return {
        "node_id": cliente,

        "in_degree": len(predecesores),
        "out_degree": len(sucesores),

        "degree_total": len(vecinos),
        "cantidad_contrapartes": len(vecinos),

        "monto_enviado": monto_enviado,
        "monto_recibido": monto_recibido,
        "monto_total": monto_enviado + monto_recibido,

        "operaciones_enviadas": operaciones_enviadas,
        "operaciones_recibidas": operaciones_recibidas
    }


In [ ]:
metricas_p1 = pd.DataFrame(
    [
        metricas_cliente(G, cliente)
        for cliente in clientes_p1_grafo
    ]
)

print(metricas_p1.shape)

metricas_p1.head()


In [ ]:
# =========================
# 10. EXPOSICIÓN A RIESGO
# =========================

def exposicion_riesgo(G, cliente):

    if cliente not in G:
        return None

    vecinos = (
        set(G.successors(cliente)) |
        set(G.predecessors(cliente))
    )

    total = len(vecinos)

    n_pep = 0
    n_ros = 0
    n_alerta = 0
    n_noticia = 0

    for vecino in vecinos:

        attrs = G.nodes[vecino]

        n_pep += int(
            attrs.get("flag_pep", 0) == 1
        )

        n_ros += int(
            attrs.get("flag_ros", 0) == 1
        )

        n_alerta += int(
            attrs.get("flag_alerta", 0) == 1
        )

        n_noticia += int(
            attrs.get("flag_noticia", 0) == 1
        )

    return {
        "node_id": cliente,

        "n_vecinos": total,

        "n_vecinos_pep": n_pep,
        "pct_vecinos_pep": n_pep / total if total else 0,

        "n_vecinos_ros": n_ros,
        "pct_vecinos_ros": n_ros / total if total else 0,

        "n_vecinos_alerta": n_alerta,
        "pct_vecinos_alerta": n_alerta / total if total else 0,

        "n_vecinos_noticia": n_noticia,
        "pct_vecinos_noticia": n_noticia / total if total else 0
    }


In [ ]:
riesgo_p1 = pd.DataFrame(
    [
        exposicion_riesgo(G, cliente)
        for cliente in clientes_p1_grafo
    ]
)

print(riesgo_p1.shape)

riesgo_p1.head()


In [ ]:
print("metricas_p1 existe:", "metricas_p1" in globals())
print("riesgo_p1 existe:", "riesgo_p1" in globals())


In [ ]:
# =========================
# 11. CREAR FEATURES P5 SCORE > 0.98
# =========================

features_p1 = metricas_p1.merge(
    riesgo_p1,
    on="node_id",
    how="left"
)

print(features_p1.shape)
features_p1.head()


In [ ]:
alertas_p1.head()


In [ ]:
columnas_p1 = [
    col
    for col in [
        "cod_cli",
        "score",
        "grupo_score",
        "target",
        "tip_alerta",
        "trx_riesgo_cliente"
    ]
    if col in p1.columns
]

info_p1 = (
    p1[columnas_p1]
    .drop_duplicates("cod_cli")
)

features_p1 = features_p1.merge(
    info_p1,
    left_on="node_id",
    right_on="cod_cli",
    how="left"
)

features_p1.drop(
    columns="cod_cli",
    inplace=True,
    errors="ignore"
)


In [ ]:
print(features_p1.shape)
print(features_p1.columns.tolist())


In [ ]:
print(features_p1.shape)

features_p1.head()


In [ ]:
print(features_p1.shape)

features_p1.head()


In [ ]:
# =========================
# 12. GRAFO NO DIRIGIDO
# =========================

G_und = G.to_undirected()

print(
    f"G_und: {G_und.number_of_nodes():,} nodos / "
    f"{G_und.number_of_edges():,} aristas"
)


In [ ]:
# =========================
# 13. PESO PARA LOUVAIN
# =========================

for u, v, data in G_und.edges(data=True):

    data["peso_louvain"] = np.log1p(
        data.get("monto_total", 0)
    )


In [ ]:
# =========================
# 14. COMUNIDADES LOUVAIN
# =========================

comunidades_louvain = nx.community.louvain_communities(
    G_und,
    weight="peso_louvain",
    resolution=1,
    seed=42
)

print(
    "Cantidad de comunidades:",
    len(comunidades_louvain)
)


In [ ]:
# =========================
# 15. MAPA COMUNIDADES
# =========================

particion = {}

for id_comunidad, miembros in enumerate(
    comunidades_louvain
):
    for nodo in miembros:
        particion[nodo] = id_comunidad


In [ ]:
nx.set_node_attributes(
    G,
    particion,
    "comunidad"
)

nx.set_node_attributes(
    G_und,
    particion,
    "comunidad"
)


In [ ]:
features_p1["comunidad"] = (
    features_p1["node_id"]
    .map(particion)
)


In [ ]:
# =========================
# 16. CENTRALIDAD GLOBAL
# =========================

n_total = G.number_of_nodes()
normalizador = max(n_total - 1, 1)

centralidad = pd.DataFrame({
    "node_id": list(G.nodes())
})

centralidad["in_degree_centrality"] = centralidad["node_id"].map(dict(G.in_degree())) / normalizador
centralidad["out_degree_centrality"] = centralidad["node_id"].map(dict(G.out_degree())) / normalizador
centralidad["degree_centrality"] = centralidad["node_id"].map(dict(G_und.degree())) / normalizador

centralidad["weighted_in_degree"] = centralidad["node_id"].map(dict(G.in_degree(weight="monto_total"))).fillna(0)
centralidad["weighted_out_degree"] = centralidad["node_id"].map(dict(G.out_degree(weight="monto_total"))).fillna(0)
centralidad["weighted_degree_total"] = (
    centralidad["weighted_in_degree"] +
    centralidad["weighted_out_degree"]
)

print(centralidad.shape)
centralidad.head()


In [ ]:
# =========================
# 17. PAGERANK PONDERADO
# =========================

pagerank = nx.pagerank(
    G,
    alpha=0.85,
    weight="monto_total",
    max_iter=100,
    tol=1e-06
)

centralidad["pagerank"] = centralidad["node_id"].map(pagerank).fillna(0)

features_p1 = features_p1.merge(
    centralidad[
        [
            "node_id",
            "degree_centrality",
            "in_degree_centrality",
            "out_degree_centrality",
            "weighted_degree_total",
            "pagerank"
        ]
    ],
    on="node_id",
    how="left"
)


In [ ]:
# =========================
# 16. STATS COMUNIDAD
# =========================

registros = []

for nodo, attrs in G.nodes(data=True):

    registros.append({
        "node_id": nodo,
        "comunidad": particion.get(nodo),

        "flag_ros": attrs.get("flag_ros", 0),
        "flag_pep": attrs.get("flag_pep", 0),
        "flag_alerta": attrs.get("flag_alerta", 0),
        "flag_noticia": attrs.get("flag_noticia", 0)
    })

df_comunidades = pd.DataFrame(registros)


In [ ]:
# =========================
# 20. LIDER DE COMUNIDAD
# =========================

centralidad["comunidad"] = centralidad["node_id"].map(particion)

lider_comunidad = (
    centralidad
    .dropna(subset=["comunidad"])
    .sort_values(
        ["comunidad", "pagerank", "weighted_degree_total"],
        ascending=[True, False, False]
    )
    .drop_duplicates("comunidad")
    [["comunidad", "node_id", "pagerank"]]
    .rename(
        columns={
            "node_id": "lider_comunidad",
            "pagerank": "lider_pagerank"
        }
    )
)

flags_lider = (
    df_comunidades[
        [
            "node_id",
            "flag_ros",
            "flag_alerta",
            "flag_pep",
            "flag_noticia"
        ]
    ]
    .rename(
        columns={
            "node_id": "lider_comunidad",
            "flag_ros": "lider_flag_ros",
            "flag_alerta": "lider_flag_alerta",
            "flag_pep": "lider_flag_pep",
            "flag_noticia": "lider_flag_noticia"
        }
    )
)

lider_comunidad = lider_comunidad.merge(
    flags_lider,
    on="lider_comunidad",
    how="left"
)

features_p1 = features_p1.merge(
    lider_comunidad,
    on="comunidad",
    how="left"
)

features_p1["pagerank_comunidad_rank"] = (
    features_p1
    .groupby("comunidad")["pagerank"]
    .rank(method="dense", ascending=False)
)


In [ ]:
stats_comunidad = (
    df_comunidades
    .groupby(
        "comunidad",
        as_index=False
    )
    .agg(
        cantidad_nodos=("node_id", "nunique"),
        n_ros=("flag_ros", "sum"),
        n_pep=("flag_pep", "sum"),
        n_alertas=("flag_alerta", "sum"),
        n_noticias=("flag_noticia", "sum")
    )
)

stats_comunidad["casos_pep_comunidad"] = stats_comunidad["n_pep"]
stats_comunidad["casos_noticias_comunidad"] = stats_comunidad["n_noticias"]


In [ ]:
for variable in [
    "ros",
    "pep",
    "alertas",
    "noticias"
]:

    stats_comunidad[f"pct_{variable}"] = (
        stats_comunidad[f"n_{variable}"] /
        stats_comunidad["cantidad_nodos"]
    )


In [ ]:
features_p1 = features_p1.merge(
    stats_comunidad,
    on="comunidad",
    how="left"
)


In [ ]:
# =========================
# 17. FLAGS COMUNIDAD
# =========================

features_p1["comunidad_con_ros"] = (
    features_p1["n_ros"] > 0
).astype("int8")

features_p1["comunidad_con_pep"] = (
    features_p1["n_pep"] > 0
).astype("int8")

features_p1["comunidad_con_alertas"] = (
    features_p1["n_alertas"] > 0
).astype("int8")


In [ ]:
# =========================
# 18. NODOS ROS
# =========================

nodos_ros = {
    nodo
    for nodo, attrs in G.nodes(data=True)
    if attrs.get("flag_ros", 0) == 1
}

print(f"Cantidad de nodos ROS: {len(nodos_ros):,}")


In [ ]:
# =========================
# 19. DISTANCIA AL ROS MÁS CERCANO
# VERSION EFICIENTE
# =========================

nodos_ros_validos = {
    nodo
    for nodo in nodos_ros
    if nodo in G_und
}

print("Nodos ROS:", len(nodos_ros))
print(
    "Nodos ROS presentes en el grafo:",
    len(nodos_ros_validos)
)


In [ ]:
nodo_virtual = "__ROS_SOURCE__"

G_und.add_node(nodo_virtual)

G_und.add_edges_from(
    (nodo_virtual, ros)
    for ros in nodos_ros_validos
)


In [ ]:
distancias = nx.single_source_shortest_path_length(
    G_und,
    nodo_virtual,
    cutoff=5
)


In [ ]:
distancias_ros = {
    nodo: distancia - 1
    for nodo, distancia in distancias.items()
    if nodo != nodo_virtual
}


In [ ]:
features_p1["distancia_ros"] = (
    features_p1["node_id"]
    .map(distancias_ros)
)


In [ ]:
G_und.remove_node(nodo_virtual)


In [ ]:
print(
    features_p1["distancia_ros"]
    .value_counts(dropna=False)
    .sort_index()
)


In [ ]:
columnas_salida = [
    "node_id",
    "grupo_score",
    "score",
    "target",

    "comunidad",
    "cantidad_nodos",

    "degree_total",
    "degree_centrality",
    "in_degree_centrality",
    "out_degree_centrality",
    "weighted_degree_total",
    "pagerank",
    "pagerank_comunidad_rank",

    "lider_comunidad",
    "lider_pagerank",
    "lider_flag_ros",
    "lider_flag_alerta",
    "lider_flag_pep",
    "lider_flag_noticia",

    "monto_enviado",
    "monto_recibido",
    "monto_total",

    "n_vecinos_ros",
    "pct_vecinos_ros",
    "distancia_ros",

    "n_ros",
    "pct_ros",

    "n_vecinos_alerta",
    "pct_vecinos_alerta",

    "n_alertas",
    "pct_alertas",

    "n_vecinos_pep",
    "pct_vecinos_pep",
    "n_pep",
    "pct_pep",
    "casos_pep_comunidad",

    "n_vecinos_noticia",
    "pct_vecinos_noticia",
    "n_noticias",
    "pct_noticias",
    "casos_noticias_comunidad"
]

vista_analista = features_p1[
    [col for col in columnas_salida if col in features_p1.columns]
].copy()

vista_analista.head()


In [ ]:
# =========================
# 24. GUARDAR TABLA FINAL
# =========================

vista_analista.to_csv(
    "vista_analista_p5_score_098.csv",
    index=False,
    encoding="utf-8-sig"
)

vista_analista.to_parquet(
    "vista_analista_p5_score_098.parquet",
    index=False
)

print("Archivos guardados:")
print("- vista_analista_p5_score_098.csv")
print("- vista_analista_p5_score_098.parquet")


In [ ]:
# =========================
# CASO A - ROS DIRECTO
# =========================

caso_a = (
    vista_analista[
        vista_analista["n_vecinos_ros"] > 0
    ]
    .sort_values(
        ["n_vecinos_ros", "monto_total"],
        ascending=[False, False]
    )
    .iloc[0]
)

caso_a


In [ ]:
# =========================
# CASO B - ROS INDIRECTO
# =========================

caso_b = (
    vista_analista[
        (vista_analista["n_vecinos_ros"] == 0) &
        (vista_analista["n_ros"] > 0) &
        (vista_analista["distancia_ros"].isin([2, 3]))
    ]
    .sort_values(
        ["distancia_ros", "n_ros", "monto_total"],
        ascending=[True, False, False]
    )
    .iloc[0]
)

caso_b


In [ ]:
cliente_a = caso_a["node_id"]
cliente_b = caso_b["node_id"]

print("Caso A:", cliente_a)
print("Caso B:", cliente_b)


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np


In [ ]:
# =========================
# 21. GRAFICO DE INVESTIGACION
# =========================

def graficar_investigacion(G, cliente, features_p1, radio=1):

    if cliente not in G:
        print(f"Cliente {cliente} no está en el grafo")
        return

    fila = features_p1.loc[
        features_p1["node_id"] == cliente
    ]

    if fila.empty:
        print("Cliente sin información en features_p1")
        return

    fila = fila.iloc[0]

    # -----------------------------------------
    # Ego network
    # -----------------------------------------

    G_und_local = G.to_undirected()

    nodos_ego = list(
        nx.single_source_shortest_path_length(
            G_und_local,
            cliente,
            cutoff=radio
        ).keys()
    )

    ego = G.subgraph(nodos_ego).copy()

    # -----------------------------------------
    # Layout
    # -----------------------------------------

    pos = nx.spring_layout(
        ego,
        seed=42,
        k=1.8
    )

    # -----------------------------------------
    # Clasificar nodos
    # -----------------------------------------

    colores = []
    tamanios = []

    for nodo in ego.nodes:

        attrs = G.nodes[nodo]

        if nodo == cliente:
            colores.append("red")
            tamanios.append(1800)

        elif attrs.get("flag_ros", 0) == 1:
            colores.append("purple")
            tamanios.append(1100)

        elif attrs.get("flag_pep", 0) == 1:
            colores.append("orange")
            tamanios.append(900)

        elif attrs.get("flag_alerta", 0) == 1:
            colores.append("gold")
            tamanios.append(800)

        elif attrs.get("flag_noticia", 0) == 1:
            colores.append("skyblue")
            tamanios.append(650)

        else:
            colores.append("lightgray")
            tamanios.append(450)

    # -----------------------------------------
    # Grosor según monto
    # -----------------------------------------

    montos = [
        data.get("monto_total", 0)
        for _, _, data in ego.edges(data=True)
    ]

    if montos:
        max_monto = max(montos)

        widths = [
            1 + 6 * (m / max_monto)
            if max_monto > 0 else 1
            for m in montos
        ]
    else:
        widths = 1

    # -----------------------------------------
    # Dibujar
    # -----------------------------------------

    plt.figure(figsize=(14, 10))

    nx.draw_networkx_nodes(
        ego,
        pos,
        node_color=colores,
        node_size=tamanios,
        alpha=0.9
    )

    nx.draw_networkx_edges(
        ego,
        pos,
        arrows=True,
        arrowstyle="-|>",
        arrowsize=18,
        width=widths,
        alpha=0.5
    )

    # etiquetas sólo para cliente y nodos de riesgo
    labels = {}

    for nodo in ego.nodes:

        attrs = G.nodes[nodo]

        if (
            nodo == cliente
            or attrs.get("flag_ros", 0) == 1
            or attrs.get("flag_pep", 0) == 1
            or attrs.get("flag_alerta", 0) == 1
        ):
            labels[nodo] = nodo

    nx.draw_networkx_labels(
        ego,
        pos,
        labels=labels,
        font_size=8
    )

    # -----------------------------------------
    # Título analítico
    # -----------------------------------------

    titulo = (
        f"Cliente P5: {cliente}\n"
        f"Comunidad: {fila.get('comunidad')} | "
        f"Tamaño comunidad: {fila.get('cantidad_nodos'):,.0f}\n"
        f"Contrapartes: {fila.get('degree_total'):,.0f} | "
        f"ROS directos: {fila.get('n_vecinos_ros'):,.0f} | "
        f"ROS comunidad: {fila.get('n_ros'):,.0f} | "
        f"Distancia ROS: {fila.get('distancia_ros')}\n"
        f"Monto total: {fila.get('monto_total'):,.0f}"
    )

    plt.title(
        titulo,
        fontsize=13
    )

    plt.axis("off")
    plt.show()


In [ ]:
graficar_investigacion(
    G,
    cliente_a,
    features_p1,
    radio=1
)


In [ ]:
graficar_investigacion(
    G,
    cliente_b,
    features_p1,
    radio=1
)


In [ ]:
casos_3 = features_p1[
    features_p1["degree_total"] == 3
].copy()

print("Cantidad de P5 score > 0.98 con 3 contrapartes:", len(casos_3))

casos_3[
    [
        "node_id",
        "comunidad",
        "monto_enviado",
        "monto_recibido",
        "monto_total",
        "n_vecinos_ros",
        "n_vecinos_alerta",
        "n_ros",
        "distancia_ros"
    ]
].sort_values(
    "monto_total",
    ascending=False
).head(20)


In [ ]:
features_p1[
    features_p1["node_id"].isin(
        ["0008264985", "0016251583"]
    )
][
    [
        "node_id",
        "comunidad",
        "cantidad_nodos",
        "in_degree",
        "out_degree",
        "degree_total",
        "monto_enviado",
        "monto_recibido",
        "monto_total",
        "n_vecinos_ros",
        "pct_vecinos_ros",
        "n_vecinos_alerta",
        "pct_vecinos_alerta",
        "n_ros",
        "pct_ros",
        "distancia_ros"
    ]
]


In [ ]:
graficar_investigacion(
    G,
    "0008264985",
    features_p1,
    radio=1
)


In [ ]:
graficar_investigacion(
    G,
    "0016251583",
    features_p1,
    radio=1
)


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np


def graficar_investigacion(
    G,
    cliente,
    features_p1,
    radio=1,
    mostrar=True
):

    if cliente not in G:
        print(f"Cliente {cliente} no está en el grafo")
        return None

    fila = features_p1.loc[
        features_p1["node_id"] == cliente
    ]

    if fila.empty:
        print(f"Cliente {cliente} sin información en features_p1")
        return None

    fila = fila.iloc[0]

    # ==============================
    # EGO NETWORK
    # ==============================

    G_local = G.to_undirected()

    nodos_ego = list(
        nx.single_source_shortest_path_length(
            G_local,
            cliente,
            cutoff=radio
        ).keys()
    )

    ego = G.subgraph(nodos_ego).copy()

    # ==============================
    # POSICIONES
    # ==============================

    pos = nx.spring_layout(
        ego,
        seed=42,
        k=1.8
    )

    # ==============================
    # NODOS
    # ==============================

    colores = []
    tamanios = []

    for nodo in ego.nodes:

        attrs = G.nodes[nodo]

        if nodo == cliente:
            colores.append("red")
            tamanios.append(1800)

        elif attrs.get("flag_ros", 0) == 1:
            colores.append("purple")
            tamanios.append(1100)

        elif attrs.get("flag_pep", 0) == 1:
            colores.append("orange")
            tamanios.append(900)

        elif attrs.get("flag_alerta", 0) == 1:
            colores.append("gold")
            tamanios.append(800)

        elif attrs.get("flag_noticia", 0) == 1:
            colores.append("skyblue")
            tamanios.append(650)

        else:
            colores.append("lightgray")
            tamanios.append(450)

    # ==============================
    # GROSOR DE ARISTAS
    # ==============================

    montos = [
        data.get("monto_total", 0)
        for _, _, data in ego.edges(data=True)
    ]

    if len(montos) > 0:

        max_monto = max(montos)

        widths = [
            1 + 6 * (m / max_monto)
            if max_monto > 0
            else 1
            for m in montos
        ]

    else:
        widths = 1

    # ==============================
    # FIGURA
    # ==============================

    fig, ax = plt.subplots(
        figsize=(14, 9)
    )

    nx.draw_networkx_nodes(
        ego,
        pos,
        node_color=colores,
        node_size=tamanios,
        alpha=0.9,
        ax=ax
    )

    nx.draw_networkx_edges(
        ego,
        pos,
        arrows=True,
        arrowstyle="-|>",
        arrowsize=18,
        width=widths,
        alpha=0.5,
        ax=ax
    )

    # ==============================
    # LABELS
    # ==============================

    labels = {}

    for nodo in ego.nodes:

        attrs = G.nodes[nodo]

        if (
            nodo == cliente
            or attrs.get("flag_ros", 0) == 1
            or attrs.get("flag_pep", 0) == 1
            or attrs.get("flag_alerta", 0) == 1
        ):
            labels[nodo] = nodo

    nx.draw_networkx_labels(
        ego,
        pos,
        labels=labels,
        font_size=8,
        ax=ax
    )

    # ==============================
    # MONTOS EN LAS ARISTAS
    # ==============================

    edge_labels = {}

    for origen, destino, attrs in ego.edges(data=True):

        monto = attrs.get("monto_total", 0)

        if monto >= 1_000_000:
            texto = f"{monto/1_000_000:.1f}M"

        elif monto >= 1_000:
            texto = f"{monto/1_000:.1f}K"

        else:
            texto = f"{monto:,.0f}"

        edge_labels[(origen, destino)] = texto

    nx.draw_networkx_edge_labels(
        ego,
        pos,
        edge_labels=edge_labels,
        font_size=8,
        ax=ax
    )

    # ==============================
    # TITULO
    # ==============================

    distancia_ros = fila.get(
        "distancia_ros",
        np.nan
    )

    titulo = (
        f"Cliente P5: {cliente}\n"
        f"Comunidad: {fila.get('comunidad')} | "
        f"Tamaño comunidad: {fila.get('cantidad_nodos'):,.0f}\n"
        f"Contrapartes: {fila.get('degree_total'):,.0f} | "
        f"ROS directos: {fila.get('n_vecinos_ros'):,.0f} | "
        f"ROS comunidad: {fila.get('n_ros'):,.0f} | "
        f"Distancia ROS: {distancia_ros}\n"
        f"Monto total: {fila.get('monto_total'):,.0f}"
    )

    ax.set_title(
        titulo,
        fontsize=13
    )

    ax.axis("off")

    plt.tight_layout()

    if mostrar:
        plt.show()

    return fig


In [ ]:
descripcion_variables = {
    "node_id": "Identificador unico del cliente investigado.",
    "grupo_score": "Grupo de priorizacion asignado por el modelo. En este analisis se trabaja sobre clientes P5 con score mayor a 0.98.",
    "score": "Score generado por el modelo de priorizacion PLAFT.",
    "target": "Marca de alerta/calificacion usada como variable objetivo o referencia operativa.",
    "comunidad": "Identificador de la comunidad detectada mediante Louvain.",
    "cantidad_nodos": "Cantidad total de clientes que forman parte de la comunidad.",
    "degree_total": "Cantidad de contrapartes unicas con las cuales el cliente mantiene relaciones transaccionales.",
    "degree_centrality": "Centralidad de grado normalizada dentro del grafo.",
    "in_degree_centrality": "Centralidad normalizada por relaciones entrantes.",
    "out_degree_centrality": "Centralidad normalizada por relaciones salientes.",
    "weighted_degree_total": "Monto transaccional total conectado al nodo; suma enviada y recibida.",
    "pagerank": "PageRank ponderado por monto_total de las aristas. Mide importancia estructural considerando conexiones entrantes relevantes.",
    "pagerank_comunidad_rank": "Ranking del cliente dentro de su comunidad segun PageRank; 1 indica mayor centralidad.",
    "lider_comunidad": "Nodo con mayor PageRank dentro de la comunidad.",
    "lider_pagerank": "PageRank del lider de la comunidad.",
    "lider_flag_ros": "Indica si el lider de la comunidad tiene ROS.",
    "lider_flag_alerta": "Indica si el lider de la comunidad tiene alerta.",
    "lider_flag_pep": "Indica si el lider de la comunidad es PEP.",
    "lider_flag_noticia": "Indica si el lider de la comunidad tiene noticias.",
    "monto_enviado": "Monto total enviado por el cliente hacia sus contrapartes.",
    "monto_recibido": "Monto total recibido por el cliente desde sus contrapartes.",
    "monto_total": "Suma del monto enviado y recibido.",
    "n_vecinos_ros": "Cantidad de contrapartes directas del cliente que poseen ROS.",
    "pct_vecinos_ros": "Porcentaje de las contrapartes directas que poseen ROS.",
    "distancia_ros": "Cantidad minima de saltos en la red hasta encontrar un nodo con ROS.",
    "n_ros": "Cantidad total de clientes con ROS dentro de la comunidad.",
    "pct_ros": "Proporcion de clientes con ROS dentro de la comunidad.",
    "n_vecinos_alerta": "Cantidad de contrapartes directas que poseen flag de alerta.",
    "pct_vecinos_alerta": "Porcentaje de contrapartes directas con flag de alerta.",
    "n_alertas": "Cantidad total de clientes con alerta dentro de la comunidad.",
    "pct_alertas": "Proporcion de clientes con alerta dentro de la comunidad.",
    "n_vecinos_pep": "Cantidad de contrapartes directas PEP.",
    "pct_vecinos_pep": "Porcentaje de contrapartes directas PEP.",
    "n_pep": "Cantidad de clientes PEP presentes dentro de la comunidad.",
    "pct_pep": "Proporcion de clientes PEP dentro de la comunidad.",
    "casos_pep_comunidad": "Cantidad de casos PEP acumulados en la comunidad.",
    "n_vecinos_noticia": "Cantidad de contrapartes directas con noticias.",
    "pct_vecinos_noticia": "Porcentaje de contrapartes directas con noticias.",
    "n_noticias": "Cantidad de clientes con noticias dentro de la comunidad.",
    "pct_noticias": "Proporcion de clientes con noticias dentro de la comunidad.",
    "casos_noticias_comunidad": "Cantidad de casos con noticias acumulados en la comunidad."
}


In [ ]:
import io
import base64


def figura_a_base64(fig):

    buffer = io.BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=130,
        bbox_inches="tight"
    )

    buffer.seek(0)

    imagen_base64 = base64.b64encode(
        buffer.read()
    ).decode("utf-8")

    buffer.close()

    plt.close(fig)

    return imagen_base64


In [ ]:
columnas_ejemplo = [
    "node_id",
    "grupo_score",
    "score",
    "comunidad",
    "cantidad_nodos",
    "degree_total",
    "pagerank",
    "pagerank_comunidad_rank",
    "lider_comunidad",
    "lider_flag_ros",
    "lider_flag_alerta",
    "monto_enviado",
    "monto_recibido",
    "monto_total",
    "n_vecinos_ros",
    "distancia_ros",
    "n_ros",
    "n_alertas",
    "casos_pep_comunidad",
    "casos_noticias_comunidad"
]


In [ ]:
tabla_ejemplo = (
    pd.concat(
        [
            vista_analista[
                vista_analista["node_id"].astype(str).isin(
                    ["0022279923", "0021447280", "0021155690"]
                )
            ],
            vista_analista.sort_values(
                ["score", "monto_total"],
                ascending=[False, False]
            ).head(7)
        ],
        ignore_index=True
    )
    .drop_duplicates("node_id")
    .head(10)
    .copy()
)


In [ ]:
for col in [
    "monto_enviado",
    "monto_recibido",
    "monto_total",
    "pagerank",
    "lider_pagerank"
]:
    if col in tabla_ejemplo.columns:
        tabla_ejemplo[col] = tabla_ejemplo[col].map(
            lambda x: f"{x:.3e}" if col in ["pagerank", "lider_pagerank"] and pd.notna(x)
            else f"{x:,.2f}" if pd.notna(x)
            else ""
        )

tabla_html = tabla_ejemplo.to_html(
    index=False,
    classes="tabla-datos tabla-compacta",
    border=0
)


In [ ]:
variables_df = pd.DataFrame(
    [
        {
            "Variable": variable,
            "Funcionalidad / interpretación": descripcion
        }
        for variable, descripcion
        in descripcion_variables.items()
    ]
)

variables_html = variables_df.to_html(
    index=False,
    classes="tabla-variables",
    border=0
)


In [ ]:
# =========================
# CASOS PARA EL REPORTE P5 SCORE > 0.98
# =========================

cliente_1 = "0022279923"
cliente_2 = "0021447280"

clientes_reporte = {cliente_1, cliente_2}
clientes_disponibles = set(vista_analista["node_id"].astype(str))
clientes_faltantes = clientes_reporte - clientes_disponibles

if clientes_faltantes:
    raise ValueError(
        f"Clientes no encontrados en vista_analista: {clientes_faltantes}"
    )

print("Caso 4:", cliente_1)
print("Caso 5:", cliente_2)


In [ ]:
fig_1 = graficar_investigacion(
    G,
    cliente_1,
    features_p1,
    radio=1,
    mostrar=False
)

fig_2 = graficar_investigacion(
    G,
    cliente_2,
    features_p1,
    radio=1,
    mostrar=False
)


In [ ]:
img_1 = figura_a_base64(fig_1)
img_2 = figura_a_base64(fig_2)


In [ ]:
def obtener_ficha(features_p1, cliente):

    fila = (
        features_p1[
            features_p1["node_id"] == cliente
        ]
        .iloc[0]
    )

    return {
        "Cliente": cliente,
        "Comunidad": fila.get("comunidad"),
        "Tamano comunidad": fila.get("cantidad_nodos"),
        "Contrapartes": fila.get("degree_total"),
        "PageRank": fila.get("pagerank"),
        "Rank PageRank comunidad": fila.get("pagerank_comunidad_rank"),
        "Lider comunidad": fila.get("lider_comunidad"),
        "Lider con ROS": fila.get("lider_flag_ros"),
        "Lider con alerta": fila.get("lider_flag_alerta"),
        "Monto enviado": fila.get("monto_enviado"),
        "Monto recibido": fila.get("monto_recibido"),
        "Monto total": fila.get("monto_total"),
        "ROS directos": fila.get("n_vecinos_ros"),
        "Distancia ROS": fila.get("distancia_ros"),
        "ROS comunidad": fila.get("n_ros"),
        "Alertas directas": fila.get("n_vecinos_alerta"),
        "Alertas comunidad": fila.get("n_alertas"),
        "Casos PEP comunidad": fila.get("casos_pep_comunidad"),
        "Casos noticias comunidad": fila.get("casos_noticias_comunidad")
    }


In [ ]:
ficha_1 = obtener_ficha(
    features_p1,
    cliente_1
)

ficha_2 = obtener_ficha(
    features_p1,
    cliente_2
)


In [ ]:
def ficha_a_html(ficha):

    filas = ""

    for clave, valor in ficha.items():

        if isinstance(valor, (float, np.floating)):

            if "Monto" in clave:
                valor = f"{valor:,.2f}"

            elif not pd.isna(valor):
                valor = f"{valor:,.2f}"

        filas += f"""
        <tr>
            <td>{clave}</td>
            <td>{valor}</td>
        </tr>
        """

    return f"""
    <table class="ficha">
        {filas}
    </table>
    """


In [ ]:
ficha_1_html = ficha_a_html(ficha_1)
ficha_2_html = ficha_a_html(ficha_2)


In [ ]:
html = f"""
<!DOCTYPE html>

<html lang="es">

<head>

<meta charset="UTF-8">

<title>Grafo Transaccional PLAFT</title>

<style>

body {{
    font-family: Arial, Helvetica, sans-serif;
    background: #f4f6f8;
    color: #1f2937;
    margin: 0;
}}

.container {{
    width: 94%;
    max-width: 1500px;
    margin: auto;
}}

.header {{
    background: linear-gradient(
        135deg,
        #0b4f6c,
        #0f766e
    );

    color: white;

    padding: 45px;

    margin-bottom: 30px;
}}

.header h1 {{
    margin: 0;
    font-size: 36px;
}}

.header p {{
    font-size: 17px;
    opacity: 0.9;
}}

.section {{
    background: white;
    padding: 30px;
    margin-bottom: 25px;
    border-radius: 12px;

    box-shadow:
        0 3px 10px
        rgba(0,0,0,0.06);
}}

h2 {{
    color: #0b4f6c;
    border-bottom: 2px solid #0f766e;
    padding-bottom: 8px;
}}

h3 {{
    color: #0f766e;
}}

.intro {{
    font-size: 16px;
    line-height: 1.6;
}}

.destacado {{
    background: #e8f3f1;
    border-left: 5px solid #0f766e;
    padding: 15px;
    margin: 20px 0;
}}

.tabla-datos,
.tabla-variables,
.ficha {{

    border-collapse: collapse;
    width: 100%;
    font-size: 13px;
}}

.tabla-datos th,
.tabla-variables th,
.ficha th {{

    background: #0b4f6c;
    color: white;
    padding: 9px;
}}

.tabla-datos td,
.tabla-variables td,
.ficha td {{

    border-bottom: 1px solid #ddd;
    padding: 8px;
}}

.tabla-datos tr:nth-child(even),
.tabla-variables tr:nth-child(even) {{

    background: #f8fafc;
}}

.tabla-compacta {{
    font-size: 10px;
    table-layout: auto;
}}

.tabla-compacta th,
.tabla-compacta td {{
    padding: 4px 5px;
    line-height: 1.15;
    white-space: nowrap;
}}

.tabla-compacta th {{
    font-size: 9px;
}}

.tabla-compacta td {{
    font-size: 10px;
}}

.tabla-compacta-wrap {{
    overflow-x: auto;
    max-width: 100%;
}}

.caso {{

    display: grid;

    grid-template-columns:
        30% 70%;

    gap: 20px;

    align-items: start;
}}

.ficha {{
    background: #fafafa;
}}

.ficha td:first-child {{
    font-weight: bold;
    color: #475467;
}}

.grafico img {{
    width: 100%;
    border-radius: 10px;
}}

.leyenda {{
    display: flex;
    gap: 20px;
    flex-wrap: wrap;
    margin-top: 20px;
}}

.item-leyenda {{
    font-size: 13px;
}}

.circulo {{
    width: 14px;
    height: 14px;
    display: inline-block;
    border-radius: 50%;
    margin-right: 5px;
}}

.rojo {{ background:red; }}
.violeta {{ background:purple; }}
.naranja {{ background:orange; }}
.amarillo {{ background:gold; }}
.celeste {{ background:skyblue; }}
.gris {{ background:lightgray; }}

@media(max-width:900px) {{

    .caso {{
        grid-template-columns: 1fr;
    }}

}}

</style>

</head>


<body>

<div class="header">

    <div class="container">

        <h1>
            Grafo Transaccional PLAFT
        </h1>

        <p>
            Análisis de clientes P5 con score mayor a 0.98,
            exposición a riesgo y
            estructura de comunidades
        </p>

    </div>

</div>


<div class="container">


<div class="section">

<h2>1. Objetivo</h2>

<p class="intro">

El objetivo del análisis es complementar
la priorización tradicional de alertas PLAFT
con información de estructura de red.

Para cada cliente P5 con score mayor a 0.98 se analiza:

</p>

<ul>

<li>
cantidad y dirección de relaciones
transaccionales;
</li>

<li>
montos enviados y recibidos;
</li>

<li>
exposición directa a clientes con ROS,
PEP, alertas o noticias;
</li>

<li>
comunidad transaccional a la cual
pertenece;
</li>

<li>
proximidad estructural a clientes
con ROS.

</li>

</ul>

</div>



<div class="section">

<h2>2. Ejemplo de salida analítica</h2>

<p>

Cada fila representa un cliente P5 con score mayor a 0.98
y consolida información del modelo,
de su red directa y de la comunidad
a la que pertenece.

</p>

<div class="tabla-compacta-wrap">

{tabla_html}

</div>

<div class="destacado">
<b>Nota sobre pagerank_comunidad_rank:</b>
este ranking se calcula dentro del universo analizado del reporte, es decir, clientes P5 con score mayor a 0.98 que estan presentes en el grafo. En cambio, <b>lider_comunidad</b> corresponde al nodo con mayor PageRank dentro de toda la comunidad Louvain, incluyendo clientes que no necesariamente estan en el universo P5 filtrado. Por eso un cliente puede tener <b>pagerank_comunidad_rank = 1</b> dentro del filtro P5 y aun asi tener otro nodo como lider de la comunidad.
</div>

</div>



<div class="section">

<h2>3. Interpretación de las variables</h2>

<p>

Las variables están diseñadas para responder
preguntas específicas durante una investigación
PLAFT.

</p>

<div style="overflow-x:auto">

{variables_html}

</div>

</div>



<div class="section">

<h2>4. Caso de investigación: {cliente_1}</h2>

<div class="caso">

<div>

<h3>Ficha del cliente</h3>

{ficha_1_html}

</div>

<div class="grafico">

<img
src="data:image/png;base64,{img_1}"
>

</div>

</div>

<div class="destacado">

Este caso permite observar un cliente P5 con score mayor a 0.98
con pocas contrapartes directas.
El grosor de las relaciones representa
la magnitud relativa del monto transaccionado.

</div>

</div>



<div class="section">

<h2>5. Caso de investigación: {cliente_2}</h2>

<div class="caso">

<div>

<h3>Ficha del cliente</h3>

{ficha_2_html}

</div>

<div class="grafico">

<img
src="data:image/png;base64,{img_2}"
>

</div>

</div>

<div class="destacado">

Este caso permite analizar simultáneamente
la relación directa del cliente y
la existencia de señales de riesgo
en sus contrapartes.

</div>

</div>



<div class="section">

<h2>6. Leyenda del grafo</h2>

<div class="leyenda">

<div class="item-leyenda">
<span class="circulo rojo"></span>
Cliente P5 investigado
</div>

<div class="item-leyenda">
<span class="circulo violeta"></span>
Cliente con ROS
</div>

<div class="item-leyenda">
<span class="circulo naranja"></span>
PEP
</div>

<div class="item-leyenda">
<span class="circulo amarillo"></span>
Cliente con alerta
</div>

<div class="item-leyenda">
<span class="circulo celeste"></span>
Cliente con noticia
</div>

<div class="item-leyenda">
<span class="circulo gris"></span>
Sin señal de riesgo identificada
</div>

</div>

</div>



<div class="section">

<h2>7. Lectura para el analista PLAFT</h2>

<div class="destacado">

<b>Riesgo directo:</b>
n_vecinos_ros,
n_vecinos_alerta,
n_vecinos_pep.

<br><br>

<b>Riesgo estructural:</b>
comunidad,
n_ros,
pct_ros,
distancia_ros.

<br><br>

<b>Comportamiento transaccional:</b>
degree_total,
monto_enviado,
monto_recibido,
monto_total.

</div>

<p>

La combinación de estos tres niveles permite
diferenciar clientes que pueden tener
un volumen elevado por su actividad normal
de aquellos que presentan estructuras
de relación especialmente relevantes
para una investigación PLAFT.

</p>

</div>


</div>

</body>

</html>
"""


In [ ]:
# =========================
# 26. AGREGAR DATOS DE CONSTRUCCION AL HTML ORIGINAL
# =========================

periodos_grafo = sorted(
    edges["periodo"].dropna().astype(str).unique().tolist()
)
periodo_grafo = ", ".join(periodos_grafo)

nodos_grafo_set = set(G.nodes())
clientes_banco_set = set(nodos["node_id"].dropna().unique())
clientes_banco_grafo = len(nodos_grafo_set & clientes_banco_set)

segmento_analizado = "BPE"
grupo_score_analizado = "P5"
periodo_score = "202607"
score_minimo = 0.98
clientes_score_filtrados = len(p1)

html_construccion = f"""

<div class="destacado">

<h3>Construccion del grafo</h3>

<p>
El grafo fue construido con operaciones del periodo <b>{periodo_grafo}</b>, tomando cada cliente como nodo y cada relacion transaccional origen-destino como arista. La base se enfoca en clientes del segmento <b>{segmento_analizado}</b> priorizados en el grupo <b>{grupo_score_analizado}</b> con <b>score &gt; {score_minimo}</b>.
</p>

<div class="bloque-riesgo">

<div class="tarjeta">
<h4>Periodo y alcance</h4>
<p>
<b>Periodo de operaciones:</b> {periodo_grafo}<br>
<b>Periodo de alertas del universo P5 filtrado:</b> 202604<br>
<b>Periodo del score:</b> {periodo_score}<br>
<b>Segmento:</b> {segmento_analizado}<br>
<b>Grupo analizado:</b> {grupo_score_analizado}<br>
<b>Filtro score:</b> &gt; {score_minimo}
</p>
</div>

<div class="tarjeta">
<h4>Tamano del grafo</h4>
<p>
<b>Nodos:</b> {G.number_of_nodes():,.0f}<br>
<b>Aristas:</b> {G.number_of_edges():,.0f}<br>
<b>Tipo de nodo:</b> cliente / contraparte<br>
<b>Tipo de arista:</b> relacion transaccional agregada origen-destino
</p>
</div>

<div class="tarjeta">
<h4>Universo de clientes</h4>
<p>
<b>P5 score &gt; {score_minimo} en score:</b> {clientes_score_filtrados:,.0f}<br>
<b>P5 score &gt; {score_minimo} presentes en grafo:</b> {len(vista_analista):,.0f}<br>
<b>Clientes en maestro PLAFT:</b> {nodos["node_id"].nunique():,.0f}<br>
<b>Clientes del banco presentes en grafo:</b> {clientes_banco_grafo:,.0f}<br>
<b>Comunidades con clientes P5 con score mayor a 0.98:</b> {vista_analista["comunidad"].nunique():,.0f}
</p>
</div>

</div>

<p>
<b>Fuentes:</b> e_perm_aws.t_aml_operaciones, e_perm_aws.t_mst_parametros_plaft, e_perm_aws.t_cliente_plaft, e_perm_aws.t_alertas_plaft, e_perm_aws.t_listas_nivel_de_riesgos_motivo_r3 y score_202607_minorista.csv.
</p>

</div>
"""

inicio_objetivo = html.find("<h2>1. Objetivo</h2>")
inicio_siguiente_seccion = html.find(
    '<div class="section">',
    inicio_objetivo + 1
)

if inicio_objetivo >= 0 and inicio_siguiente_seccion >= 0:
    bloque_objetivo = html[inicio_objetivo:inicio_siguiente_seccion]

    if "Construccion del grafo" not in bloque_objetivo:
        posicion_insercion = html.rfind(
            "</div>",
            inicio_objetivo,
            inicio_siguiente_seccion
        )

        html = (
            html[:posicion_insercion] +
            html_construccion +
            "\n" +
            html[posicion_insercion:]
        )

archivo_salida = "reporte_grafos_plaft.html"


In [ ]:
with open(
    archivo_salida,
    "w",
    encoding="utf-8"
) as f:

    f.write(html)

print(
    f"HTML generado correctamente: "
    f"{archivo_salida}"
)


In [ ]:
import webbrowser
import os

webbrowser.open(
    "file://" +
    os.path.abspath(
        archivo_salida
    )
)


In [ ]:
# =========================
# 27. HTML PARA PRESENTACION: CASOS EN COMUNIDADES PEQUENAS
# =========================

metric_cols = [
    "n_ros",
    "n_alertas",
    "casos_pep_comunidad",
    "casos_noticias_comunidad",
    "n_vecinos_ros",
    "n_vecinos_alerta",
    "n_vecinos_pep",
    "n_vecinos_noticia"
]

for col in metric_cols:
    if col not in vista_analista.columns:
        vista_analista[col] = 0

casos_presentacion = vista_analista[
    vista_analista["cantidad_nodos"].between(2, 1000)
].copy()

casos_presentacion["signal_score"] = (
    casos_presentacion["n_ros"].fillna(0) * 6 +
    casos_presentacion["n_alertas"].fillna(0) * 4 +
    casos_presentacion["casos_pep_comunidad"].fillna(0) * 3 +
    casos_presentacion["casos_noticias_comunidad"].fillna(0) +
    casos_presentacion["n_vecinos_ros"].fillna(0) * 8 +
    casos_presentacion["n_vecinos_alerta"].fillna(0) * 4 +
    casos_presentacion["pagerank_comunidad_rank"].fillna(999).rsub(1000).clip(lower=0) * 0.01
)

casos_presentacion = (
    casos_presentacion
    .sort_values(
        ["signal_score", "pagerank", "monto_total"],
        ascending=[False, False, False]
    )
    .drop_duplicates("comunidad")
    .head(8)
    .copy()
)

columnas_presentacion = [
    "node_id",
    "comunidad",
    "cantidad_nodos",
    "degree_total",
    "pagerank_comunidad_rank",
    "lider_comunidad",
    "lider_flag_ros",
    "lider_flag_alerta",
    "monto_total",
    "n_vecinos_ros",
    "distancia_ros",
    "n_ros",
    "n_alertas",
    "casos_pep_comunidad",
    "casos_noticias_comunidad"
]

columnas_presentacion = [
    col for col in columnas_presentacion
    if col in casos_presentacion.columns
]

casos_presentacion[columnas_presentacion].to_csv(
    "casos_presentacion_comunidades_pequenas.csv",
    index=False,
    encoding="utf-8-sig"
)


def formato_presentacion(valor, columna):
    if pd.isna(valor):
        return ""
    if columna in ["monto_total"]:
        return f"{valor:,.2f}"
    if columna in ["pagerank", "lider_pagerank"]:
        return f"{valor:,.8f}"
    if isinstance(valor, (int, np.integer)) or (
        isinstance(valor, float) and float(valor).is_integer()
    ):
        return f"{valor:,.0f}"
    if isinstance(valor, float):
        return f"{valor:,.2f}"
    return str(valor)


def nombre_columna_presentacion(columna):
    nombres = {
        "node_id": "Cliente",
        "comunidad": "Comunidad",
        "cantidad_nodos": "Nodos comunidad",
        "degree_total": "Contrapartes",
        "pagerank_comunidad_rank": "Rank PR comunidad",
        "lider_comunidad": "Lider comunidad",
        "lider_flag_ros": "Lider ROS",
        "lider_flag_alerta": "Lider alerta",
        "monto_total": "Monto total",
        "n_vecinos_ros": "ROS directos",
        "distancia_ros": "Dist. ROS",
        "n_ros": "ROS comunidad",
        "n_alertas": "Alertas comunidad",
        "casos_pep_comunidad": "PEP comunidad",
        "casos_noticias_comunidad": "Noticias comunidad"
    }
    return nombres.get(columna, columna)


def tabla_presentacion_html(df_tabla):
    encabezados = "".join(
        f"<th>{nombre_columna_presentacion(col)}</th>"
        for col in df_tabla.columns
    )

    filas = []
    for _, fila in df_tabla.iterrows():
        celdas = "".join(
            f"<td>{formato_presentacion(fila[col], col)}</td>"
            for col in df_tabla.columns
        )
        filas.append(f"<tr>{celdas}</tr>")

    return f"""
    <table class="tabla-datos">
        <thead><tr>{encabezados}</tr></thead>
        <tbody>{''.join(filas)}</tbody>
    </table>
    """


def badge(valor, positivo="Si", negativo="No"):
    try:
        valor = int(valor)
    except Exception:
        valor = 0

    clase = "yes" if valor == 1 else "no"
    texto = positivo if valor == 1 else negativo
    return f'<span class="badge {clase}">{texto}</span>'


tabla_casos_html = tabla_presentacion_html(
    casos_presentacion[columnas_presentacion]
)

cards = []
for _, fila in casos_presentacion.iterrows():
    distancia = formato_presentacion(
        fila.get("distancia_ros"),
        "distancia_ros"
    ) or "Sin ROS cercano"

    cards.append(f"""
    <article class="case-card">
        <div class="case-head">
            <div>
                <p class="eyebrow">Comunidad {formato_presentacion(fila.get('comunidad'), 'comunidad')} ? {formato_presentacion(fila.get('cantidad_nodos'), 'cantidad_nodos')} nodos</p>
                <h3>{fila.get('node_id')}</h3>
            </div>
            <div class="rank">Rank PR<br><b>{formato_presentacion(fila.get('pagerank_comunidad_rank'), 'pagerank_comunidad_rank')}</b></div>
        </div>
        <div class="metrics">
            <div><b>{formato_presentacion(fila.get('monto_total'), 'monto_total')}</b><span>Monto total</span></div>
            <div><b>{formato_presentacion(fila.get('degree_total'), 'degree_total')}</b><span>Contrapartes</span></div>
            <div><b>{formato_presentacion(fila.get('n_ros'), 'n_ros')}</b><span>ROS comunidad</span></div>
            <div><b>{formato_presentacion(fila.get('n_alertas'), 'n_alertas')}</b><span>Alertas comunidad</span></div>
        </div>
        <p class="readout">
            Lider: <b>{fila.get('lider_comunidad')}</b> {badge(fila.get('lider_flag_ros'), 'ROS', 'sin ROS')} {badge(fila.get('lider_flag_alerta'), 'alerta', 'sin alerta')}
        </p>
        <p class="readout">
            PEP comunidad: <b>{formato_presentacion(fila.get('casos_pep_comunidad'), 'casos_pep_comunidad')}</b> ? Noticias comunidad: <b>{formato_presentacion(fila.get('casos_noticias_comunidad'), 'casos_noticias_comunidad')}</b> ? Distancia ROS: <b>{distancia}</b>
        </p>
    </article>
    """)

cards_html = "\n".join(cards)

html_presentacion = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<title>Casos para Presentacion - Grafo PLAFT</title>
<style>
body {{ font-family: Arial, Helvetica, sans-serif; background:#f4f6f8; color:#1f2937; margin:0; }}
.container {{ width:94%; max-width:1450px; margin:auto; }}
.header {{ background:linear-gradient(135deg,#0b4f6c,#0f766e); color:white; padding:38px 0; margin-bottom:26px; }}
.header h1 {{ margin:0; font-size:32px; }}
.header p {{ margin:10px 0 0; font-size:16px; opacity:.92; }}
.section {{ background:white; padding:26px; margin-bottom:22px; border-radius:10px; box-shadow:0 3px 10px rgba(0,0,0,.06); }}
h2 {{ color:#0b4f6c; border-bottom:2px solid #0f766e; padding-bottom:8px; margin-top:0; }}
h3 {{ margin:3px 0 0; color:#12364a; }}
.note {{ background:#e8f3f1; border-left:5px solid #0f766e; padding:14px; line-height:1.5; }}
.kpis {{ display:grid; grid-template-columns:repeat(4,1fr); gap:14px; margin-top:16px; }}
.kpi {{ background:#f8fafc; border:1px solid #d9e1e7; border-radius:8px; padding:14px; }}
.kpi b {{ display:block; color:#0b4f6c; font-size:12px; text-transform:uppercase; }}
.kpi span {{ font-size:25px; font-weight:700; }}
.tabla-datos {{ border-collapse:collapse; width:100%; font-size:12px; }}
.tabla-datos th {{ background:#0b4f6c; color:white; padding:8px; white-space:nowrap; }}
.tabla-datos td {{ border-bottom:1px solid #ddd; padding:7px; text-align:right; white-space:nowrap; }}
.tabla-datos td:first-child, .tabla-datos td:nth-child(6) {{ text-align:left; }}
.tabla-datos tr:nth-child(even) {{ background:#f8fafc; }}
.case-grid {{ display:grid; grid-template-columns:repeat(2,1fr); gap:16px; }}
.case-card {{ border:1px solid #d9e1e7; border-radius:8px; padding:16px; background:#fbfcfe; }}
.case-head {{ display:flex; justify-content:space-between; gap:12px; align-items:flex-start; border-bottom:1px solid #e5e7eb; padding-bottom:10px; }}
.eyebrow {{ margin:0; color:#64748b; font-size:12px; text-transform:uppercase; }}
.rank {{ min-width:82px; text-align:center; background:#e8f3f1; border-radius:8px; padding:8px; color:#0b4f6c; font-size:12px; }}
.rank b {{ font-size:20px; }}
.metrics {{ display:grid; grid-template-columns:repeat(4,1fr); gap:10px; margin:14px 0; }}
.metrics div {{ background:white; border:1px solid #edf0f2; border-radius:8px; padding:10px; }}
.metrics b {{ display:block; font-size:15px; color:#0b4f6c; }}
.metrics span {{ display:block; color:#64748b; font-size:11px; margin-top:4px; }}
.readout {{ margin:8px 0 0; font-size:13px; line-height:1.45; }}
.badge {{ display:inline-block; border-radius:999px; padding:3px 8px; margin-left:6px; font-size:11px; font-weight:700; }}
.badge.yes {{ background:#fee2e2; color:#991b1b; }}
.badge.no {{ background:#e5e7eb; color:#475569; }}
@media(max-width:950px) {{ .case-grid, .kpis {{ grid-template-columns:1fr; }} .metrics {{ grid-template-columns:repeat(2,1fr); }} }}
</style>
</head>
<body>
<div class="header"><div class="container">
<h1>Casos seleccionados para presentacion</h1>
<p>Clientes P5 con score mayor a 0.98 en comunidades peque?as, distintas entre si, priorizados por se?ales de ROS, alertas, PEP, noticias, PageRank y monto.</p>
</div></div>
<div class="container">
<section class="section">
<h2>1. Criterio de seleccion</h2>
<div class="note">Se seleccionaron 8 casos de comunidades con 2 a 1,000 nodos. Cada caso pertenece a una comunidad distinta para evitar repetir la misma estructura en la presentacion.</div>
<div class="kpis">
<div class="kpi"><b>Casos</b><span>{len(casos_presentacion):,}</span></div>
<div class="kpi"><b>Comunidades</b><span>{casos_presentacion['comunidad'].nunique():,}</span></div>
<div class="kpi"><b>Comunidad menor</b><span>{casos_presentacion['cantidad_nodos'].min():,.0f}</span></div>
<div class="kpi"><b>Comunidad mayor</b><span>{casos_presentacion['cantidad_nodos'].max():,.0f}</span></div>
</div>
</section>
<section class="section">
<h2>2. Ejemplo de salida analitica</h2>
<div style="overflow-x:auto">{tabla_casos_html}</div>
</section>
<section class="section">
<h2>3. Lectura ejecutiva de casos</h2>
<div class="case-grid">{cards_html}</div>
</section>
</div>
</body>
</html>
"""

archivo_presentacion = "reporte_grafos_plaft_presentacion_casos.html"

with open(
    archivo_presentacion,
    "w",
    encoding="utf-8"
) as f:
    f.write(html_presentacion)

print(f"HTML de presentacion generado: {archivo_presentacion}")
print("CSV de casos generado: casos_presentacion_comunidades_pequenas.csv")
